In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
import pandas as pd
import re

chrome_driver_path = 'D:\Project\Liga 1 Indonesia\Liga-1-Indonesia\chromedriver-win64\chromedriver.exe'


# Setup Driver
options = webdriver.ChromeOptions()
#options.add_argument("--headless") # Biar Chrome-nya tidak dibuka (Jalan di belakang)
service = Service(chrome_driver_path)
driver = webdriver.Chrome(service=service, options=options)
driver.maximize_window() # Kalo headless, ini nggak berguna

link_klub = pd.read_csv('Link_Klub.csv')
link_klub = [link + "/squad" for link in link_klub['Link Klub'].tolist()]


for link in link_klub[:2]:
    driver.get(link)
    time.sleep(5)

    pattern = r'team-([a-zA-Z\-]+)'

    # Mencocokkan pola pada URL
    match = re.search(pattern, link)
    if match:
        nama_klub = match.group(1).replace('-', ' ').title()  # Mengganti '-' dengan spasi dan mengubah huruf pertama jadi kapital
        print("Nama Klub:", nama_klub)
    else:
        print("Nama klub tidak ditemukan.")

    pemain_links = []

    a_elements = driver.find_elements(By.XPATH, '//div[contains(@class, "player-item") or contains(@class, "player-info")]/a')

    # Kalau cara di atas gak berhasil, kita cari semua <a> yang pattern href-nya mengarah ke pemain
    if not a_elements:
        a_elements = driver.find_elements(By.XPATH, '//a[contains(@href, "/player-")]')

    for a in a_elements:
        link = a.get_attribute("href")
        if link and link not in pemain_links:
            pemain_links.append(link)

    # Simpan ke DataFrame dan ekspor ke CSV
    df_links = pd.DataFrame(pemain_links, columns=["Link Pemain"])
    df_links.to_csv(f"Link Pemain/{nama_klub}.csv", index=False)

    print(f"Berhasil ambil {len(pemain_links)} link pemain.")

driver.quit()

Nama Klub: Persib Bandung
Berhasil ambil 32 link pemain.
Nama Klub: Dewa United Fc
Berhasil ambil 29 link pemain.
